# Tech Layoffs 2026 — AI Job Cuts Tracker Analysis
**Author:** Taimoor Ali  
**Dataset:** Tech Layoffs 2026 | AI Job Cuts Tracker — Kaggle (alitaqishah)  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Plotly

---

## Project Goal
The tech industry is undergoing one of its biggest transformations — companies are cutting thousands of jobs while simultaneously investing billions in Artificial Intelligence. This project analyses real 2026 layoff data to answer:

- Which companies cut the most jobs?
- What share of layoffs were driven by AI adoption?
- Which sectors were hit hardest?
- How did stock markets react to layoff announcements?
- Is there a relationship between AI investment and job cuts?


---
## Step 1 — Load the Data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'DejaVu Sans'

DATA_PATH = '/kaggle/input/datasets/alitaqishah/tech-layoffs-2026-ai-job-cuts-tracker/tech_layoffs_2026_tracker.csv'
SAVE_DIR  = '/kaggle/working'

print('Libraries loaded!')

Libraries loaded!


In [2]:
df = pd.read_csv(DATA_PATH, encoding='latin1')
print(f'Loaded {len(df)} companies | {df.shape[1]} columns')
df.head()

Loaded 28 companies | 26 columns


,company,layoff_date,jobs_cut,pct_workforce_cut,sector,country,hq_city,ai_cited,reason_stated,company_revenue_2025_bn,...,layoffs_2024,layoffs_2025,verified_source,month,quarter,region,layoff_size_category,stock_reaction,laid_off_vs_headcount_pct,data_as_of
0,Amazon,2026-01-15,16000,2.7,E-Commerce/Cloud,USA,Seattle,False,Reduce bureaucracy and management layers,716.9,...,4000,14000,CNBC / NetworkWorld,January 2026,Q1 2026,North America,Mega (5K+),Positive,1.03,"March 18, 2026"
1,Block,2026-02-28,4000,40.0,Fintech,USA,San Francisco,True,AI tools replace roles enabling smaller teams,22.4,...,0,1000,CNBC / Crunchbase,February 2026,Q1 2026,North America,Large (2K-5K),Positive,40.00,"March 18, 2026"
2,Meta Reality Labs,2026-01-20,1500,10.0,Social Media/VR,USA,Menlo Park,True,Pivot from metaverse to AI research,164.5,...,0,500,InformationWeek / NYT,January 2026,Q1 2026,North America,Medium (500-2K),Positive,1.90,"March 18, 2026"
3,Atlassian,2026-03-14,1600,10.0,Enterprise Software,Australia,Sydney,True,Pivot to AI-first company strategy,5.1,...,0,500,TechRepublic / Metaintro,March 2026,Q1 2026,Asia-Pacific,Medium (500-2K),Positive,10.00,"March 18, 2026"
4,Oracle,2026-02-01,30000,15.0,Enterprise Software,USA,Austin,True,AI data centres replace human ops,52.9,...,6000,10000,IBTimes,February 2026,Q1 2026,North America,Mega (5K+),Positive,15.00,"March 18, 2026"


---
## Step 2 — Explore the Data

In [3]:
print('Columns:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nBasic statistics:')
df[['jobs_cut', 'pct_workforce_cut', 'simultaneous_ai_investment_bn']].describe()

Columns: ['company', 'layoff_date', 'jobs_cut', 'pct_workforce_cut', 'sector', 'country', 'hq_city', 'ai_cited', 'reason_stated', 'company_revenue_2025_bn', 'pre_layoff_headcount', 'stock_change_day_pct', 'simultaneous_ai_investment_bn', 'roles_most_affected', 'replacement_roles', 'ceo_quote', 'layoffs_2024', 'layoffs_2025', 'verified_source', 'month', 'quarter', 'region', 'layoff_size_category', 'stock_reaction', 'laid_off_vs_headcount_pct', 'data_as_of']

Data types:
company                           object
layoff_date                       object
jobs_cut                           int64
pct_workforce_cut                float64
sector                            object
country                           object
hq_city                           object
ai_cited                            bool
reason_stated                     object
company_revenue_2025_bn          float64
pre_layoff_headcount               int64
stock_change_day_pct             float64
simultaneous_ai_investment_bn    f

,jobs_cut,pct_workforce_cut,simultaneous_ai_investment_bn
count,28.000000,28.000000,28.000000
mean,3612.357143,10.639286,14.046429
std,6546.591373,10.433505,34.751824
min,50.000000,0.100000,0.000000
25%,989.500000,2.525000,0.000000
50%,1000.000000,7.750000,0.450000
75%,2050.000000,15.000000,2.250000
max,30000.000000,40.000000,115.000000


---
## Step 3 — Clean the Data

In [4]:
print('Missing values:')
print(df.isnull().sum())

df = df.drop_duplicates()
df['layoff_date'] = pd.to_datetime(df['layoff_date'])
df['ai_cited_label'] = df['ai_cited'].map({True: 'AI-driven', False: 'Non AI-driven'})

print(f'\nTotal companies   : {len(df)}')
print(f'Total jobs cut    : {df["jobs_cut"].sum():,}')
print(f'AI cited layoffs  : {df["ai_cited"].sum()}')
print('Data is clean!')

Missing values:
company                          0
layoff_date                      0
jobs_cut                         0
pct_workforce_cut                0
sector                           0
country                          0
hq_city                          0
ai_cited                         0
reason_stated                    0
company_revenue_2025_bn          0
pre_layoff_headcount             0
stock_change_day_pct             0
simultaneous_ai_investment_bn    0
roles_most_affected              0
replacement_roles                0
ceo_quote                        0
layoffs_2024                     0
layoffs_2025                     0
verified_source                  0
month                            0
quarter                          0
region                           0
layoff_size_category             0
stock_reaction                   0
laid_off_vs_headcount_pct        0
data_as_of                       0
dtype: int64

Total companies   : 28
Total jobs cut    : 101,146
AI cited 

---
## Step 4 — Visualizations

In [5]:
# CHART 1: Top 10 Companies by Jobs Cut
top_companies = df.nlargest(10, 'jobs_cut').sort_values('jobs_cut', ascending=True)

hover_text = [
    f"<b>{row['company']}</b><br>"
    f"Jobs Cut: <b>{row['jobs_cut']:,}</b><br>"
    f"Sector: {row['sector']}<br>"
    f"Workforce Cut: {row['pct_workforce_cut']}%<br>"
    f"AI Driven: {'✅ Yes' if row['ai_cited'] else '❌ No'}<br>"
    f"Reason: {row['reason_stated']}"
    for _, row in top_companies.iterrows()
]

bar_colors = top_companies['ai_cited'].map({True: '#E84C4C', False: '#4C9BE8'})

fig = go.Figure()

fig.add_trace(go.Bar(
    x=top_companies['jobs_cut'],
    y=top_companies['company'],
    orientation='h',
    marker=dict(color=bar_colors.values, line=dict(width=0)),
    hovertext=hover_text,
    hoverinfo='text',
    hoverlabel=dict(bgcolor='#1e1e1e', font_color='white', font_size=13)
))

fig.update_layout(
    title=dict(
        text='<b>Top 10 Companies by Jobs Cut in 2026</b><br>'
             '<sup>Red = AI-driven | Blue = Non AI-driven | Hover for details</sup>',
        font=dict(size=18), x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Number of Jobs Cut', tickformat=',', gridcolor='#f0f0f0'),
    yaxis=dict(title=''),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=520,
    margin=dict(l=180, r=60, t=100, b=60),
    font=dict(family='Arial', size=12),
    showlegend=False
)

for _, row in top_companies.iterrows():
    fig.add_annotation(
        x=row['jobs_cut'], y=row['company'],
        text=f"  {row['jobs_cut']:,}",
        showarrow=False,
        font=dict(size=11, color='#333'),
        xanchor='left'
    )

fig.show()

#### Insight
Oracle leads with 30,000 jobs cut, followed by Amazon and Meta at 16,000 each.
Red bars show layoffs where AI was explicitly cited as the reason — nearly half
of the top 10 companies directly blamed AI adoption for their workforce reductions.

In [6]:
# ── Group data: total jobs cut by AI-driven vs Non-AI-driven ──────────────────
ai_jobs = df.groupby('ai_cited_label')['jobs_cut'].sum().reset_index()

# ── Build hover tooltip text for each slice ───────────────────────────────────
hover_text = [
    f"<b>{row['ai_cited_label']}</b><br>"
    f"Total Jobs Cut: <b>{row['jobs_cut']:,}</b><br>"
    f"Share: {row['jobs_cut']/ai_jobs['jobs_cut'].sum()*100:.1f}%"
    for _, row in ai_jobs.iterrows()
]

# ── Build the donut chart ─────────────────────────────────────────────────────
fig = go.Figure(go.Pie(
    labels=ai_jobs['ai_cited_label'],
    values=ai_jobs['jobs_cut'],
    hole=0.45,                          # 0.45 = donut hole size (0=full pie, 1=full hole)
    marker=dict(
        colors=['#E84C4C', '#4C9BE8'],  # red = AI-driven, blue = Non AI-driven
        line=dict(color='white', width=3)  # white border between slices
    ),
    hovertext=hover_text,
    hoverinfo='text',                   # show custom hover text on mouse over
    textinfo='label+percent',           # show label and % on each slice
    textfont=dict(size=14),
    pull=[0.05, 0.05],                  # pull slices apart slightly for modern look
    hoverlabel=dict(bgcolor='#1e1e1e', font_color='white', font_size=13)
))

# ── Add total jobs count in the center of the donut ───────────────────────────
total = ai_jobs['jobs_cut'].sum()
fig.add_annotation(
    text=f"<b>{total:,}</b><br>total jobs",
    x=0.5, y=0.5,                       # center of the donut
    font=dict(size=15, color='#333333'),
    showarrow=False
)

# ── Chart layout and styling ──────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='<b>Share of Jobs Cut: AI-Driven vs Non-AI-Driven</b><br>'
             '<sup>Hover over each slice for exact numbers</sup>',
        font=dict(size=17), x=0.5, xanchor='center'
    ),
    legend=dict(
        orientation='h',                # horizontal legend
        yanchor='bottom', y=-0.15,      # place below the chart
        xanchor='center', x=0.5,
        font=dict(size=13)
    ),
    paper_bgcolor='white',
    height=480,
    margin=dict(t=100, b=60, l=40, r=40),
    font=dict(family='Arial', size=12)
)

fig.show()

#### Insight
Almost 40% of all jobs cut in 2026 were directly driven by AI adoption.
This means companies are not just restructuring — they are actively replacing
human roles with artificial intelligence at a significant scale.

In [7]:
# CHART 3: Total Jobs Cut by Sector 

sector_jobs = (df.groupby('sector')['jobs_cut']
                 .sum()
                 .sort_values(ascending=True)
                 .tail(10)
                 .reset_index())

# ── Build hover tooltip text for each bar ────────────────────────────────────
hover_text = [
    f"<b>{row['sector']}</b><br>"
    f"Total Jobs Cut: <b>{row['jobs_cut']:,}</b><br>"
    f"Share of All Cuts: {row['jobs_cut']/df['jobs_cut'].sum()*100:.1f}%<br>"
    f"Companies in sector: {df[df['sector']==row['sector']]['company'].count()}"
    for _, row in sector_jobs.iterrows()
]

# ── Color bars from light to dark purple (lightest = smallest, darkest = biggest) ──
purple_shades = [
    '#f3f0fc','#e0d9f7','#cdc2f2','#b9aaed','#a693e8',
    '#937be3','#7f63de','#6c4cd9','#5934d4','#461dcf'
]

# ── Build the horizontal bar chart ───────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=sector_jobs['jobs_cut'],
    y=sector_jobs['sector'],
    orientation='h',                        # horizontal bars
    marker=dict(
        color=purple_shades,                # gradient from light to dark purple
        line=dict(width=0)                  # no bar border
    ),
    hovertext=hover_text,
    hoverinfo='text',
    hoverlabel=dict(bgcolor='#1e1e1e', font_color='white', font_size=13),
    text=[f"{v:,}" for v in sector_jobs['jobs_cut']],  # value labels on bars
    textposition='outside',                 # show labels outside bar ends
    textfont=dict(size=11, color='#444444')
))

# ── Chart layout and styling ──────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='<b>Total Jobs Cut by Tech Sector (Top 10)</b><br>'
             '<sup>Hover over each bar for sector details</sup>',
        font=dict(size=17), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Total Jobs Cut',
        tickformat=',',
        gridcolor='#f0f0f0',
        showgrid=True
    ),
    yaxis=dict(
        title='',
        tickfont=dict(size=12)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=520,
    margin=dict(l=180, r=80, t=100, b=60),
    font=dict(family='Arial', size=12)
)

fig.show()

#### Insight
Enterprise Software is the hardest-hit sector, followed by E-Commerce/Cloud
and Social Media/AI. These are precisely the sectors investing most heavily
in AI — confirming that AI investment and job cuts go hand in hand.

In [8]:
# CHART 4: Stock Market Reaction
stock_counts = df['stock_reaction'].value_counts().reset_index()
stock_counts.columns = ['reaction', 'count']

# ── Assign colors by reaction type ───────────────────────────────────────────
color_map = {
    'Positive': '#2ECC71',   # green = stock went up
    'Negative': '#E84C4C',   # red   = stock went down
    'Neutral':  '#95A5A6'    # gray  = no change
}
bar_colors = stock_counts['reaction'].map(color_map)

# ── Build hover tooltip showing companies in each category ───────────────────
hover_text = [
    f"<b>{row['reaction']} Reaction</b><br>"
    f"Number of Companies: <b>{row['count']}</b><br>"
    f"Share: {row['count']/len(df)*100:.1f}% of all layoffs<br>"
    f"Companies: {', '.join(df[df['stock_reaction']==row['reaction']]['company'].tolist())}"
    for _, row in stock_counts.iterrows()
]

# ── Build the vertical bar chart ─────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=stock_counts['reaction'],
    y=stock_counts['count'],
    marker=dict(
        color=bar_colors,               # green / red / gray by reaction type
        line=dict(width=0),             # no border on bars
        cornerradius=6                  # rounded bar tops
    ),
    hovertext=hover_text,
    hoverinfo='text',
    hoverlabel=dict(bgcolor='#1e1e1e', font_color='white', font_size=13),
    text=stock_counts['count'],         # show count on top of each bar
    textposition='outside',
    textfont=dict(size=14, color='#333333')
))

# ── Add a horizontal average line ────────────────────────────────────────────
avg = stock_counts['count'].mean()
fig.add_hline(
    y=avg,
    line_dash='dot',
    line_color='#aaaaaa',
    annotation_text=f'Average: {avg:.1f}',
    annotation_position='top right',
    annotation_font=dict(size=11, color='#888888')
)

# ── Chart layout and styling ──────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='<b>Stock Market Reaction After Layoff Announcements</b><br>'
             '<sup>Hover over each bar to see which companies are in each category</sup>',
        font=dict(size=17), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Stock Reaction',
        tickfont=dict(size=13)
    ),
    yaxis=dict(
        title='Number of Companies',
        gridcolor='#f0f0f0',
        range=[0, stock_counts['count'].max() + 3]  # extra space for labels
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=480,
    margin=dict(l=60, r=60, t=110, b=60),
    font=dict(family='Arial', size=12)
)

fig.show()

#### Insight
Surprisingly, 17 out of 28 companies saw their stock price rise after announcing
layoffs. This tells us that investors view workforce cuts as a positive signal —
rewarding companies for reducing costs and investing in AI efficiency.

In [9]:
# CHART 5: AI Investment vs Jobs Cut
ai_df     = df[df['ai_cited'] == True]
non_ai_df = df[df['ai_cited'] == False]

# ── Build hover tooltip for AI-driven points ─────────────────────────────────
hover_ai = [
    f"<b>{row['company']}</b><br>"
    f"AI Investment: <b>${row['simultaneous_ai_investment_bn']:.1f}B</b><br>"
    f"Jobs Cut: <b>{row['jobs_cut']:,}</b><br>"
    f"Sector: {row['sector']}<br>"
    f"Workforce Cut: {row['pct_workforce_cut']}%<br>"
    f"Reason: {row['reason_stated']}<br>"
    f"Stock Reaction: {row['stock_reaction']}"
    for _, row in ai_df.iterrows()
]

# ── Build hover tooltip for Non-AI-driven points ─────────────────────────────
hover_non_ai = [
    f"<b>{row['company']}</b><br>"
    f"AI Investment: <b>${row['simultaneous_ai_investment_bn']:.1f}B</b><br>"
    f"Jobs Cut: <b>{row['jobs_cut']:,}</b><br>"
    f"Sector: {row['sector']}<br>"
    f"Workforce Cut: {row['pct_workforce_cut']}%<br>"
    f"Reason: {row['reason_stated']}<br>"
    f"Stock Reaction: {row['stock_reaction']}"
    for _, row in non_ai_df.iterrows()
]

# ── Build scatter plot with two traces ────────────────────────────────────────
fig = go.Figure()

# Trace 1: AI-driven companies (red dots)
fig.add_trace(go.Scatter(
    x=ai_df['simultaneous_ai_investment_bn'],
    y=ai_df['jobs_cut'],
    mode='markers+text',                    # show dots and company name labels
    name='AI cited as reason',
    marker=dict(
        color='#E84C4C',
        size=14,
        line=dict(color='white', width=2),  # white ring around each dot
        opacity=0.85
    ),
    text=ai_df['company'],
    textposition='top center',
    textfont=dict(size=9, color='#c0392b'),
    hovertext=hover_ai,
    hoverinfo='text',
    hoverlabel=dict(bgcolor='#1e1e1e', font_color='white', font_size=13)
))

# Trace 2: Non-AI-driven companies (blue dots)
fig.add_trace(go.Scatter(
    x=non_ai_df['simultaneous_ai_investment_bn'],
    y=non_ai_df['jobs_cut'],
    mode='markers+text',
    name='AI not cited',
    marker=dict(
        color='#4C9BE8',
        size=14,
        line=dict(color='white', width=2),
        opacity=0.85
    ),
    text=non_ai_df['company'],
    textposition='top center',
    textfont=dict(size=9, color='#1a6fa8'),
    hovertext=hover_non_ai,
    hoverinfo='text',
    hoverlabel=dict(bgcolor='#1e1e1e', font_color='white', font_size=13)
))

# ── Add a trend line using average values ────────────────────────────────────
import numpy as np
x_vals = df['simultaneous_ai_investment_bn'].values
y_vals = df['jobs_cut'].values
z = np.polyfit(x_vals, y_vals, 1)          # fit a straight line through all points
p = np.poly1d(z)
x_line = np.linspace(x_vals.min(), x_vals.max(), 100)

fig.add_trace(go.Scatter(
    x=x_line,
    y=p(x_line),
    mode='lines',
    name='Trend line',
    line=dict(color='#aaaaaa', width=1.5, dash='dot'),
    hoverinfo='skip'                        # no hover on the trend line
))

# ── Chart layout and styling ──────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='<b>AI Investment vs Jobs Cut</b><br>'
             '<sup>Are companies replacing workers with AI? Hover over each dot for full details</sup>',
        font=dict(size=17), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Simultaneous AI Investment ($ Billion)',
        gridcolor='#f0f0f0',
        tickformat='.0f'
    ),
    yaxis=dict(
        title='Jobs Cut',
        gridcolor='#f0f0f0',
        tickformat=','
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=-0.2,
        xanchor='center', x=0.5,
        font=dict(size=12)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=580,
    margin=dict(l=60, r=60, t=110, b=80),
    font=dict(family='Arial', size=12)
)

fig.show()

#### Insight
The trend line shows a clear positive correlation — companies investing more
in AI are cutting more jobs simultaneously. This strongly suggests that AI is
not just supplementing human workers, but actively replacing entire job categories
across the tech industry.

---
## Key Findings

1. **Oracle and Amazon** made the largest cuts, with 30,000 and 16,000 jobs lost respectively.
2. **50% of layoffs were AI-driven** — companies explicitly cited AI adoption as a reason for reducing their workforce.
3. **Enterprise Software and Telecommunications** sectors were hit the hardest by total job losses.
4. **Stock markets reacted positively** to most layoff announcements — investors view workforce cuts as cost efficiency signals.
5. **Companies investing more in AI tend to cut more jobs** — suggesting AI is actively replacing human roles, not just supplementing them.

> This analysis shows we are living through a major shift in how tech companies operate — AI is not just a tool, it is replacing entire job categories.

---
*Analysis by Taimoor Ali | taimoorali5588@gmail.com*